In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

save_dir = "data/xenium-tcr-kidney/processed"

import numpy as np
import pandas as pd
import scanpy as sc

# Set the verbosity level
sc.settings.verbosity = 3

In [2]:
path = "data/xenium/processed/03-kidney_tcr_classified.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()
# adata.obs["sample"] = [name.split("_output-")[-1] for name in adata.obs_names]
adata

AnnData object with n_obs × n_vars = 510139 × 480
    obs: 'sample', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'condition', 'cc', 'cell_type_no_tcr', 'cell_type_no_tcr_prob'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'graph'
    obsm: 'cell_type_no_tcr_probs', 'spatial'
    layers: 'counts'

In [10]:
cutoff = 1
cd8_pos_mask = np.array(
    adata[:, ["CD8A"]].layers["counts"].sum(axis=1) > cutoff
).flatten()
cd4_pos_mask = np.array(
    adata[:, ["CD4"]].layers["counts"].sum(axis=1) > cutoff
).flatten()

adata.obs["CD4_status"] = "CD4-"
adata.obs.loc[cd4_pos_mask, "CD4_status"] = "CD4+"
adata.obs["CD8_status"] = "CD8-"
adata.obs.loc[cd8_pos_mask, "CD8_status"] = "CD8+"

In [11]:
pd.crosstab(adata.obs["CD8_status"], adata.obs["CD4_status"])

CD4_status,CD4+,CD4-
CD8_status,,
CD8+,385,4200
CD8-,22427,483127
